# NaijaABSA: collect Google Play reviews

Run each cell from top to bottom (click ▶ or press **Shift + Enter**). No GPU is needed.

**What this notebook does**
1. Uses a fixed list of nine Nigerian apps
2. Downloads recent reviews (reviewer names are never saved)
3. Removes personal details, duplicates and very short or long reviews
4. Keeps reviews containing Nigerian Pidgin markers
5. Samples up to 300 per domain and downloads the files to your computer

## 1. Install the library

In [ ]:
!pip install -q google-play-scraper

## 2. The apps

Nine Nigerian apps across four domains. To add an app, open its Google Play page and copy the part after `id=` in the web address.

In [ ]:
APPS = {
    "telecoms":    ["ng.mtn.nextgen",                         # myMTN NG
                    "net.one97.selfcare.globacom"],           # Glo Cafe Nigeria
    "banking":     ["com.firstbank.firstmobile",              # FirstMobile (First Bank)
                    "com.moniepoint.business"],               # Moniepoint Business
    "electricity": ["com.powerlight.ng",                      # Powerlight
                    "com.ionicframework.buypowerapp334891",   # BuyPower
                    "com.vatebrapay.fcmbenergy"],             # Ikeja Electric
    "ecommerce":   ["com.jumia.android",                      # Jumia
                    "ng.jiji.app"],                           # Jiji Nigeria
}
MAX_PAGES = 50   # 50 x 200 = up to 10,000 reviews per app (use 10 for a quick test)

## 3. Download the reviews
With `MAX_PAGES = 50` this takes about 30–60 minutes. Keep this tab open.

In [ ]:
import time
import pandas as pd
from google_play_scraper import reviews, Sort

rows = []
for domain, app_ids in APPS.items():
    for app_id in app_ids:
        token, n = None, 0
        for _ in range(MAX_PAGES):
            try:
                batch, token = reviews(app_id, lang="en", country="ng", sort=Sort.NEWEST,
                                       count=200, continuation_token=token)
            except Exception as e:
                print("  stopped", app_id, "-", e); break
            for r in batch:                         # reviewer name is NOT saved
                rows.append({"review_id": r["reviewId"], "app_id": app_id, "domain": domain,
                             "text": r["content"], "stars": r["score"], "date": str(r["at"])})
            n += len(batch)
            if not batch or token is None:
                break
            time.sleep(2)                           # be polite to the server
        print(f"{domain:12s} {app_id:45s} {n} reviews")

raw = pd.DataFrame(rows)
raw.to_json("playstore_raw.jsonl", orient="records", lines=True)
print("Total:", len(raw))

## 4. Clean, filter for Pidgin and sample

In [ ]:
import re

df = raw.copy()

def scrub(t):
    t = re.sub(r"\S+@\S+", "[EMAIL]", str(t))
    t = re.sub(r"(\+?234|0)[789][01]\d{8}", "[PHONE]", t)
    t = re.sub(r"@\w+", "[USER]", t)
    t = re.sub(r"https?://\S+", "[URL]", t)
    return t.strip()

df["text"] = df["text"].map(scrub)
df = df.drop_duplicates("text")
df["n_words"] = df["text"].str.split().str.len()
df = df[(df.n_words >= 5) & (df.n_words <= 80)]

MARKERS = ["dey", "wey", "abeg", "no be", "una", "wetin", "sabi", "oga", "wahala",
           "comot", "abi", "sha", "shey", "e don", "don dey", "na im", "nawa", "chop",
           "o jare", "biko", "dem", "wan", "make una", "e no"]
pattern = r"\b(" + "|".join(re.escape(m) for m in MARKERS) + r")\b"
df["pidgin_hits"] = df["text"].str.lower().str.count(pattern)

print("After cleaning:", len(df))
print("With at least 1 Pidgin marker:", (df.pidgin_hits >= 1).sum())
print("With at least 2 Pidgin markers:", (df.pidgin_hits >= 2).sum())
print()
print(df[df.pidgin_hits >= 1].groupby("domain").size())

### Check by eye
Read the sample below. If many are ordinary English, change `MIN_HITS` to 2 in the next cell.

In [ ]:
pd.set_option("display.max_colwidth", 200)
df[df.pidgin_hits >= 1].sample(min(30, (df.pidgin_hits >= 1).sum()), random_state=1)[["domain", "text"]]

In [ ]:
MIN_HITS = 1          # change to 2 if too many English-only reviews
PER_DOMAIN = 300

pidgin = df[df.pidgin_hits >= MIN_HITS]
sample = pd.concat([g.sample(min(len(g), PER_DOMAIN), random_state=42)
                    for _, g in pidgin.groupby("domain")])
sample[["review_id", "app_id", "domain", "text"]].to_json(
    "to_annotate.jsonl", orient="records", lines=True)
sample[["review_id", "app_id", "domain", "text"]].to_csv("to_annotate.csv", index=False)
print(sample.domain.value_counts())

## 5. Download the files

- `to_annotate.jsonl` – import this into Label Studio (Step 3 of the guide)
- `to_annotate.csv` – the same posts, easy to open in Excel
- `playstore_raw.jsonl` – everything collected; keep it private and **do not** upload it to GitHub

Also write down today's date and the app list in your lab notebook. The paper needs them.

In [ ]:
from google.colab import files
for f in ["to_annotate.jsonl", "to_annotate.csv", "playstore_raw.jsonl"]:
    files.download(f)